# 03 - Figure 2

Creates the reported relationship between review activity and normalized degree centrality in the Hidalgo product spaces.


In [ ]:
# --- config ---
HIDALGO_RCA_INPUT = "counts"  # "counts" | "binary"
HIDALGO_SOURCE = "saved_or_compute"  # "saved_or_compute" | "compute"
HIDALGO_GRAPH = "plot_graph"  # "plot_graph" | "full_phi" | "disparity_backbone"
# Hidalgo: degree_centrality | weighted_degree_phi | betweenness | eigenvector_centrality
Y_METRIC = "degree_centrality"

import os
import sys
import importlib
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from adjustText import adjust_text
from matplotlib.patches import Patch

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
import dnm_viz_utils as dvu
dvu = importlib.reload(dvu)

repo_root = dvu.repo_root
load_master_drugs = dvu.load_master_drugs
load_category2_most_common_mapping = dvu.load_category2_most_common_mapping
load_category_color_map = dvu.load_category_color_map
MARKETS = dvu.MARKETS
MARKET_FILE_NAMES = dvu.MARKET_FILE_NAMES
style_axes = dvu.style_axes

os.chdir(repo_root())
ROOT = repo_root()
OUT = ROOT / "prescription_reports/new_category2_networks"
TAB = OUT / "tables"
NET = OUT / "networks"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
NET.mkdir(parents=True, exist_ok=True)

PANEL_LABELS = {
    "WhiteHouse Market": "A",
    "Dark0de": "B",
    "ToRReZ": "C",
    "Versus": "D",
}

df_all = load_master_drugs()
df_all = df_all[df_all["Website"].isin(MARKETS)]
prod_to_cat2 = load_category2_most_common_mapping(df_all)
color_map = load_category_color_map(prod_to_cat2)


def gcc(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G
    return G if nx.is_connected(G) else G.subgraph(max(nx.connected_components(G), key=len)).copy()


def load_saved_graphml(path: Path) -> nx.Graph | None:
    if not path.exists():
        return None
    G = nx.read_graphml(path)
    return gcc(nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}))






# --- Hidalgo φ construction (minimal copy of script 52) ---
def add_listing_id(df: pd.DataFrame) -> pd.Series:
    pages = df["Pages"] if "Pages" in df.columns else pd.Series(index=df.index, dtype=object)
    pn = df["Productname"] if "Productname" in df.columns else pd.Series(index=df.index, dtype=object)
    mask = pages.notna() & (pages.astype(str).str.strip() != "") & (pages.astype(str).str.lower() != "nan")
    return pages.where(mask, pn).astype(str).str.strip()


def vendor_product_listing_counts(df_market: pd.DataFrame) -> pd.DataFrame:
    d = df_market.dropna(subset=["ID_Num", "Category 3"]).copy()
    d["Category 3"] = d["Category 3"].astype(str).str.strip()
    d["_lid"] = add_listing_id(d)
    d = d[d["_lid"].notna() & (d["_lid"] != "")]
    agg = d.groupby(["ID_Num", "Category 3"], sort=False)["_lid"].nunique().reset_index(name="x_vp")
    return agg[agg["x_vp"] > 0]


def counts_to_matrix(counts: pd.DataFrame):
    vendors = counts["ID_Num"].astype(str).str.strip().unique().tolist()
    products = counts["Category 3"].unique().tolist()
    v_i = {v: i for i, v in enumerate(vendors)}
    p_j = {p: j for j, p in enumerate(products)}
    X = np.zeros((len(vendors), len(products)), dtype=np.float64)
    for _, row in counts.iterrows():
        X[v_i[str(row["ID_Num"]).strip()], p_j[row["Category 3"]]] = float(row["x_vp"])
    return X, vendors, products


def prepare_rca_input(X: np.ndarray, mode: str = HIDALGO_RCA_INPUT) -> np.ndarray:
    if mode == "counts":
        return X.astype(np.float64)
    if mode == "binary":
        return (X > 0).astype(np.float64)
    raise ValueError("HIDALGO_RCA_INPUT must be 'counts' or 'binary'")


def hidalgo_suffix(mode: str = HIDALGO_RCA_INPUT) -> str:
    return "" if mode == "counts" else f"_{mode}"


def rca_and_M(X: np.ndarray):
    row_sum = X.sum(axis=1, keepdims=True)
    col_sum = X.sum(axis=0, keepdims=True)
    grand = X.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        RCA = np.where((row_sum > 0) & (col_sum > 0) & (grand > 0), X * grand / (row_sum * col_sum), 0.0)
    return RCA, (RCA >= 1.0).astype(np.int8)


def phi_proximity(M: np.ndarray) -> np.ndarray:
    col_sums = M.sum(axis=0).astype(np.float64)
    MTM = (M.astype(np.float64).T @ M.astype(np.float64))
    with np.errstate(divide="ignore", invalid="ignore"):
        p_ij = np.divide(MTM, col_sums[np.newaxis, :], out=np.zeros_like(MTM), where=col_sums[np.newaxis, :] > 0)
        p_ji = np.divide(MTM, col_sums[:, np.newaxis], out=np.zeros_like(MTM), where=col_sums[:, np.newaxis] > 0)
    phi = np.minimum(p_ij, p_ji)
    np.fill_diagonal(phi, 0.0)
    for i in range(phi.shape[0]):
        if col_sums[i] > 0:
            phi[i, i] = 1.0
    return phi


def phi_to_graph(products: List[str], phi: np.ndarray) -> nx.Graph:
    G = nx.Graph()
    G.add_nodes_from(products)
    n = len(products)
    for i in range(n):
        for j in range(i + 1, n):
            w = float(phi[i, j])
            if w > 0:
                G.add_edge(products[i], products[j], weight=w, phi=w)
    return G


def load_phi_edgelist_gcc(tag: str) -> nx.Graph | None:
    if HIDALGO_SOURCE == "compute":
        return None
    suffix = hidalgo_suffix()
    p = TAB / f"{tag}_hidalgo{suffix}_phi_edgelist.csv"
    if not p.exists():
        return None
    G = nx.Graph()
    for _, r in pd.read_csv(p).iterrows():
        w = float(r["phi"])
        G.add_edge(str(r["product_i"]), str(r["product_j"]), weight=w, phi=w)
    return gcc(G)


def build_phi_gcc_from_master(df_market: pd.DataFrame) -> nx.Graph | None:
    c = vendor_product_listing_counts(df_market)
    if c.empty:
        return None
    X_counts, _, products = counts_to_matrix(c)
    X = prepare_rca_input(X_counts)
    _, M = rca_and_M(X)
    phi = phi_proximity(M)
    G = phi_to_graph(products, phi)
    return gcc(G)


def count_reviews(df_m: pd.DataFrame) -> Dict[str, int]:
    return {str(k): int(v) for k, v in df_m["Category 3"].dropna().astype(str).str.strip().value_counts().items()}




def get_hidalgo_graph(market: str, df_m: pd.DataFrame, tag: str) -> nx.Graph | None:
    suffix = hidalgo_suffix()
    if HIDALGO_SOURCE != "compute":
        G = load_saved_graphml(NET / f"{tag}_hidalgo{suffix}_{HIDALGO_GRAPH}.graphml")
        if G is not None:
            return G
        if HIDALGO_GRAPH != "full_phi":
            raise FileNotFoundError(
                f"Missing saved Hidalgo graph for {market}: {NET / f'{tag}_hidalgo{suffix}_{HIDALGO_GRAPH}.graphml'}; run notebook 02 with matching settings."
            )
        G = load_phi_edgelist_gcc(tag)
        if G is not None and G.number_of_nodes() > 0:
            return G
    return build_phi_gcc_from_master(df_m)




def rows_hidalgo(G: nx.Graph, rev: Dict[str, int]) -> pd.DataFrame:
    dc = nx.degree_centrality(G)
    bc = nx.betweenness_centrality(G, weight=None, normalized=True)
    try:
        ec = nx.eigenvector_centrality(G, weight="weight", max_iter=500)
    except Exception:
        ec = {n: np.nan for n in G.nodes()}
    rows = []
    for n in G.nodes():
        r = int(rev.get(str(n), 0))
        if r <= 0:
            continue
        wd = sum(float(G[n][nb].get("phi", G[n][nb].get("weight", 0))) for nb in G[n])
        c2 = prod_to_cat2.get(str(n), "Unknown")
        ev = ec.get(n, np.nan)
        rows.append(
            {
                "product_cat3": str(n),
                "category2": c2,
                "reviews": r,
                "degree_centrality": float(dc.get(n, 0.0)),
                "weighted_degree_phi": float(wd),
                "betweenness": float(bc.get(n, 0.0)),
                "eigenvector_centrality": float(ev) if ev == ev else np.nan,
            }
        )
    return pd.DataFrame(rows)


# --- build data for Figure 2 ---
if Y_METRIC not in ("degree_centrality", "weighted_degree_phi", "betweenness", "eigenvector_centrality"):
    raise ValueError("Invalid Y_METRIC for the Hidalgo product space")
ycol = Y_METRIC

plot_data: Dict[str, pd.DataFrame] = {}
for market in MARKETS:
    tag = MARKET_FILE_NAMES[market]
    df_m = df_all[df_all["Website"] == market].copy()
    graph = get_hidalgo_graph(market, df_m, tag)
    frame = rows_hidalgo(graph, count_reviews(df_m))
    if not frame.empty:
        plot_data[market] = frame
        print(market, len(frame), "points")

if not plot_data:
    raise RuntimeError("No plot data. Run notebook 01 first.")
print(f"Hidalgo product space: {HIDALGO_RCA_INPUT}/{HIDALGO_GRAPH}")



# --- plotting: script 54 (Hidalgo) ---
def scatter_2x2_like_54(
    plot_data: Dict[str, pd.DataFrame],
    color_map: dict,
    ycol: str,
    ylabel: str,
    fname: str,
    top_by_y: int = 4,
    top_by_reviews: int = 3,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor="white")
    axes = axes.flatten()
    for ax, market in zip(axes, MARKETS):
        dfp = plot_data.get(market, pd.DataFrame())
        if dfp.empty:
            ax.text(0.5, 0.5, f"No data for {market}", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        dfp = dfp[dfp["category2"] != "Unknown"].copy()
        if dfp.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        dfp = dfp.reset_index(drop=True)
        if ycol == "eigenvector_centrality":
            dfp = dfp.dropna(subset=[ycol])
        if dfp.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        for cat, grp in dfp.groupby("category2"):
            ax.scatter(
                grp["reviews"],
                grp[ycol],
                s=25,
                alpha=0.7,
                c=[color_map.get(cat, "#B3B3B3")],
                edgecolors="black",
                linewidths=0.2,
            )
        top_y_idx = dfp[ycol].nlargest(min(top_by_y, len(dfp))).index.tolist()
        review_candidates = dfp.drop(index=top_y_idx, errors="ignore")
        top_rev_idx = (
            review_candidates["reviews"].nlargest(min(top_by_reviews, len(review_candidates))).index.tolist()
            if len(review_candidates) > 0
            else []
        )
        label_idx = list(dict.fromkeys(top_y_idx + top_rev_idx))
        x_points = dfp["reviews"].to_numpy(dtype=float)
        y_points = dfp[ycol].to_numpy(dtype=float)
        y_max, y_min = float(dfp[ycol].max()), float(dfp[ycol].min())
        y_span = y_max - y_min
        y_pad_frac = 0.065 if market in ("Dark0de", "Versus") else 0.04
        y_pad = y_pad_frac * (y_span if y_span > 0 else max(abs(y_max), 1e-9))
        texts = []
        for j, i in enumerate(label_idx):
            row = dfp.loc[i]
            px, py = float(row["reviews"]), float(row[ycol])
            if i in top_y_idx and i not in top_rev_idx:
                tx, ty = px * 0.88, py + y_pad
            elif i in top_rev_idx and i not in top_y_idx:
                tx, ty = px * 1.12, max(0.0, py - y_pad)
            else:
                tx, ty = px * 1.05, py + 0.5 * y_pad
            if market in ("Dark0de", "Versus"):
                ty += (j % 5 - 2) * 0.022 * max(y_span, 1e-9)
                tx *= 1.0 + (j % 3 - 1) * 0.06
            prod = str(row["product_cat3"]).strip()
            if market == "Versus" and prod == "Amphetamine":
                tx, ty = px * 1.30, py + 0.48 * y_pad
            elif market == "Dark0de" and prod == "Alprazolam":
                tx *= 1.12
            name = str(row["product_cat3"])[:32]
            texts.append(ax.text(tx, ty, name, fontsize=11, fontweight="bold" if row["category2"] == "Prescription" else "normal", color="black", alpha=0.92, ha="center", va="center"))
        if texts:
            tgt_x = [float(dfp.loc[i, "reviews"]) for i in label_idx]
            tgt_y = [float(dfp.loc[i, ycol]) for i in label_idx]
            if market in ("Dark0de", "Versus"):
                adj_kw = dict(
                    expand_points=(2.9, 2.9),
                    expand_text=(2.55, 2.55),
                    force_points=(1.15, 1.15),
                    force_text=(1.2, 1.2),
                    lim=1600,
                )
            else:
                adj_kw = dict(
                    expand_points=(2.2, 2.2),
                    expand_text=(1.8, 1.8),
                    force_points=(0.9, 0.9),
                    force_text=(0.7, 0.7),
                    lim=800,
                )
            adjust_text(
                texts,
                x=x_points,
                y=y_points,
                target_x=tgt_x,
                target_y=tgt_y,
                ax=ax,
                arrowprops=dict(arrowstyle="-", color="#333333", lw=0.45, shrinkA=0, shrinkB=0),
                **adj_kw,
            )
        ax.set_xscale("log")
        ax.set_xlabel("Number of reviews", fontsize=13, color="black")
        ax.set_ylabel(ylabel, fontsize=13, color="black")
        ax.set_title(market, loc="left", fontsize=14, fontweight="normal", color="black")
        ax.text(
            -0.08,
            1.02,
            PANEL_LABELS[market],
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=18,
            fontweight="normal",
            color="black",
        )
        style_axes(ax)
    categories_present = sorted(
        [c for c in color_map.keys() if str(c).strip() not in {"", "Unknown"}],
        key=lambda c: (c != "Prescription", c),
    )
    legend_handles = [
        Patch(facecolor=color_map.get(cat, "#B3B3B3"), edgecolor="black", linewidth=0.3, label=cat)
        for cat in categories_present
    ]
    fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.01), frameon=False, fontsize=10, ncol=7)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    out_path = FIG / fname
    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print("Saved", out_path)


YLABELS_HIDALGO = {
    "degree_centrality": ("Degree centrality", f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_degree_vs_reviews_scatter.png"),
    "weighted_degree_phi": (r"Weighted degree ($\sum \phi$)", f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_weighted_degree_vs_reviews_scatter.png"),
    "betweenness": ("Betweenness centrality", f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_betweenness_vs_reviews_scatter.png"),
    "eigenvector_centrality": ("Eigenvector centrality", f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_eigenvector_vs_reviews_scatter.png"),
}

ylab, fname = YLABELS_HIDALGO[ycol]
scatter_2x2_like_54(plot_data, color_map, ycol, ylab, fname)

